<a href="https://colab.research.google.com/github/sjayavelu73/langgraph/blob/lang1/RAG_with_React_with_langgraph_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
#!pip install langgraph
from langgraph.graph import START,END,StateGraph
from langgraph.prebuilt import ToolNode
#!pip install langchain_openai
#!pip install langchain_community
#!pip install langchain_chroma
#!pip install PyMUPDF
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.colab import userdata
from google.colab import drive
from langchain_core.tools  import tool
from typing import TypedDict,Sequence,Union,Annotated
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage,BaseMessage
from langgraph.graph.message import add_messages
import fitz
from langchain_core.documents import Document
import os

os.environ["OPENAI_API_KEY"] = userdata.get('ai_agents_openai')
drive.mount('/content/drive')
file_path="/content/drive/MyDrive/RAG.pdf"
doc=fitz.open(file_path)

# Extract text per page
page_texts = [doc[page_num].get_text() for page_num in range(len(doc))]

# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# Container for Document objects
all_docs = []

# Split per page and convert to Document
for i, page_text in enumerate(page_texts):
    chunks = text_splitter.split_text(page_text)
    docs = [Document(page_content=chunk, metadata={"page": i+1}) for chunk in chunks]
    all_docs.extend(docs)

from langchain.embeddings import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()
db=Chroma.from_documents(all_docs,embeddings)

@tool
def retrive_from_pdf(query:str)->str:
  """Search the pdf for information on RAG"""
  retriever=db.as_retriever()
  docs=retriever.get_relevant_documents(query)
  return docs[0].page_content

@tool
def add_numbers(a:int, b:int)->int:
  """ Tool to add numbers"""
  return a +b


@tool
def subtract_numbers(a:int, b:int)->int:
  """ Tool to subtract numbers"""
  return a - b

@tool
def multiply_numbers(a:int, b:int)->int:
  """ Tool to multiply numbers"""
  return a * b

tools=[add_numbers,subtract_numbers,multiply_numbers,retrive_from_pdf]

llm=ChatOpenAI(temperature=0,model="gpt-4o").bind_tools(tools)

class ReasonBot(TypedDict):
  messages: Annotated[Sequence[BaseMessage],add_messages]

def model_invoke(state:ReasonBot)->ReasonBot:
  prompt="""
              You are a helpful assistant who will be answering my queries . You will intelligently use tools to find answers and
               if the tools are not relevant , you will answer independently
         """

  response = llm.invoke([prompt] + state["messages"])
  return {"messages": [response]}

def should_continue(state: ReasonBot):
  msgs=state["messages"]
  last_message=msgs[-1]
  if not last_message.tool_calls:
    return "end"
  else: return "continue"

graph=StateGraph(ReasonBot)
graph.add_node("model_invoke",model_invoke)
toolnode=ToolNode(tools)
graph.add_node("tools",toolnode)
graph.add_edge(START,"model_invoke")
graph.add_conditional_edges("model_invoke",
                            should_continue,
                            {"continue":"tools","end":END}
)
graph.add_edge("tools","model_invoke")
app=graph.compile()

inputs={"messages":[("user","Integral of log5x , what is RAG , Derivative of sin(3logx) , capital of netherlands , Add 13 to 25,Multiply 20 with 15 , Subtract 100 from 200 , Integral of logx, 89+100")]}

msg=app.stream(inputs,stream_mode="values")

for s in msg:
  print(type(s["messages"]), len(s["messages"]))
  message=s["messages"][-1]
  if isinstance(message,tuple):
    print(message)
  else:
    message.pretty_print()













Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
<class 'list'> 1
================================ Human Message =================================

Integral of log5x , what is RAG , Derivative of sin(3logx) , capital of netherlands , Add 13 to 25,Multiply 20 with 15 , Subtract 100 from 200 , Integral of logx, 89+100
<class 'list'> 2
================================== Ai Message ==================================
Tool Calls:
  retrive_from_pdf (call_CujPBcqOy2wigw53rl0CgoQl)
 Call ID: call_CujPBcqOy2wigw53rl0CgoQl
  Args:
    query: RAG
  add_numbers (call_8aS4sIFNr4aePD3EeJ8MHhUq)
 Call ID: call_8aS4sIFNr4aePD3EeJ8MHhUq
  Args:
    a: 13
    b: 25
  multiply_numbers (call_Z18IlVF5WPsgXT7juNMq825k)
 Call ID: call_Z18IlVF5WPsgXT7juNMq825k
  Args:
    a: 20
    b: 15
  subtract_numbers (call_fPOQQ8HYyC2gk3757abGl8c8)
 Call ID: call_fPOQQ8HYyC2gk3757abGl8c8
  Args:
    a: 200
    b: 100
  add_numbers (call_dN3p